# NB_hospital_daily_summary — Gold Layer: Hospital Daily Summary

Builds `caresyncdata.gold.hospital_daily_summary` — one row per hospital per calendar day with test volume, unique patient/doctor counts, abnormality rates, and day-over-day trend.

**STTM:** Silver → Gold | CareSync Health Network 
**Load type:** Full rebuild — `CREATE OR REPLACE TABLE … AS SELECT` 
**Grain:** One row per hospital per calendar day 
**Driving table:** `caresyncdata.silver.lab_results` (grouped by `hospital_id`, `test_date`) 
**Joins:** LEFT JOIN `hospitals`, LEFT JOIN `calendar` (optional — for `day_name`, `is_weekend`) 
**Window:** `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date`

In [0]:
# ---------------------------------------------------------------------------
# Configuration — substituting the <catalog> placeholder from the STTM
# ---------------------------------------------------------------------------
catalog = "caresyncdata"
silver  = f"{catalog}.silver"
gold    = f"{catalog}.gold"
target_table = f"{gold}.hospital_daily_summary"

print(f"Catalog  : {catalog}")
print(f"Source   : {silver}")
print(f"Target   : {target_table}")

In [0]:
# Ensure the gold schema exists before writing
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
print(f"Schema '{gold}' is ready.")

In [0]:
# ---------------------------------------------------------------------------
# Full rebuild — CREATE OR REPLACE TABLE ... AS SELECT
# Grain: one row per hospital per calendar day
# Driving table: silver.lab_results  (grouped by hospital_id, test_date)
# Joins: LEFT JOIN hospitals, LEFT JOIN calendar (day_name, is_weekend)
# Window: LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)
# ---------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH base_agg AS (
    -- Aggregate lab_results per hospital per day
    SELECT
        hospital_id,
        test_date                                                              AS summary_date,
        COUNT(lab_result_id)                                                   AS total_tests,
        COUNT(DISTINCT patient_id)                                             AS unique_patients_tested,
        COUNT(DISTINCT doctor_id)                                              AS active_doctor_count,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)                AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)                AS critical_test_count
    FROM {silver}.lab_results
    GROUP BY hospital_id, test_date
),
with_rates AS (
    -- Derived rate columns
    SELECT
        b.*,
        CAST(b.abnormal_test_count / NULLIF(b.total_tests, 0) * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
        CAST(b.critical_test_count / NULLIF(b.total_tests, 0) * 100 AS DECIMAL(5,2)) AS critical_rate_pct
    FROM base_agg AS b
),
with_lag AS (
    -- Day-over-day trend: LAG per hospital, ordered by date
    SELECT
        w.*,
        LAG(w.total_tests) OVER (
            PARTITION BY w.hospital_id
            ORDER BY w.summary_date
        ) AS prev_day_total_tests
    FROM with_rates AS w
)
SELECT
    wl.hospital_id,
    h.hospital_name,
    wl.summary_date,

    -- Calendar enrichment (optional)
    cal.day_name,
    cal.is_weekend,

    -- Aggregated metrics
    wl.total_tests,
    wl.unique_patients_tested,
    wl.active_doctor_count,
    wl.abnormal_test_count,
    wl.critical_test_count,
    wl.abnormal_rate_pct,
    wl.critical_rate_pct,

    -- Day-over-day trend
    wl.prev_day_total_tests,
    CAST(
        CASE
            WHEN wl.prev_day_total_tests IS NULL
              OR wl.prev_day_total_tests = 0  THEN NULL
            ELSE (wl.total_tests - wl.prev_day_total_tests)
                 / wl.prev_day_total_tests * 100
        END
    AS DECIMAL(6,2)) AS test_volume_change_pct

FROM with_lag                          AS wl
LEFT JOIN {silver}.hospitals           AS h   ON wl.hospital_id = h.hospital_id
LEFT JOIN {silver}.calendar            AS cal
    ON cal.date_key = CAST(DATE_FORMAT(wl.summary_date, 'yyyyMMdd') AS INT)
ORDER BY wl.hospital_id, wl.summary_date
""")
print(f"\u2705  {target_table} rebuilt successfully.")

In [0]:
# Quick row-count and sample check
row_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {target_table}").collect()[0]['cnt']
print(f"Total rows in {target_table}: {row_count}")

df = spark.table(target_table)
display(df.limit(10))

In [0]:
records_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {target_table}").collect()[0]['cnt']
dbutils.notebook.exit(str(records_count))